<a href="https://colab.research.google.com/github/Minhaz-ck/rugpull-detector-/blob/main/notebook/fetch_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install requests pandas -q

In [5]:

import requests

ETHERSCAN_KEY = "B74QPRTXM4WNH9UI5JRJJDG2Q4USWGX3KG"

def get_dexscreener_data(token_address: str) -> dict:
    url = f"https://api.dexscreener.com/latest/dex/tokens/{token_address}"
    r = requests.get(url, timeout=10).json()
    pairs = r.get("pairs") or []
    if not pairs:
        return {}
    p = pairs[0]  # highest-liquidity pair is usually first
    return {
        "liquidity_usd": p.get("liquidity", {}).get("usd", 0),
        "fdv": p.get("fdv", 0),
        "market_cap": p.get("marketCap", 0),
        "price_change_24h": p.get("priceChange", {}).get("h24", 0),
        "buys_24h": p.get("txns", {}).get("h24", {}).get("buys", 0),
        "sells_24h": p.get("txns", {}).get("h24", {}).get("sells", 0),
        "volume_24h": p.get("volume", {}).get("h24", 0),
        "pair_age_days": p.get("pairCreatedAt", 0),
    }

def get_etherscan_data(token_address: str) -> dict:
    holder_url = (f"https://api.etherscan.io/v2/api?chainid=1&module=token"
                  f"&action=tokenholdercount&contractaddress={token_address}&apikey={ETHERSCAN_KEY}")
    source_url = (f"https://api.etherscan.io/v2/api?chainid=1&module=contract"
                  f"&action=getsourcecode&address={token_address}&apikey={ETHERSCAN_KEY}")
    holders = requests.get(holder_url, timeout=10).json()
    source = requests.get(source_url, timeout=10).json()
    is_verified = 1
    try:
        is_verified = 0 if source["result"][0]["SourceCode"] == "" else 1
    except Exception:
        pass
    holder_count = 0
    try:
        holder_count = int(holders.get("result", 0))
    except Exception:
        pass
    return {"holder_count": holder_count, "is_verified": is_verified}

def build_features(token_address: str) -> dict:
    d = {"address": token_address}
    d.update(get_dexscreener_data(token_address))
    d.update(get_etherscan_data(token_address))
    return d